In [ ]:
from supabase import create_client
from datetime import datetime, timezone
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns


SUPABASE_URL     = os.environ["SUPABASE_URL"]
SUPABASE_KEY     = os.environ["SUPABASE_KEY"]
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [ ]:
def select_supabase(table, columns="*", filters=None, batch_size=1000):
     all_rows = []
     start = 0

     while True:
         query = supabase.table(table).select(columns).range(start, start + batch_size - 1)

         if filters:
             for f in filters:
                 query = query.filter(*f)

         resp = query.execute()
         rows = resp.data or []

         if not rows:
             break

         all_rows.extend(rows)

         if len(rows) < batch_size:
             break

         start += batch_size

     return all_rows

all_rows = select_supabase("servo_prices", columns="*", filters=None, batch_size=1000)
dfprice = pd.DataFrame(all_rows)
print(dfprice.shape)
dfprice['updated_at'] = pd.to_datetime(dfprice['updated_at'])
dfprice['updated_at_dt'] = dfprice['updated_at'].dt.date
dfprice.head()

In [ ]:
# 1st May 2026 to 1st Jul 2026 -> increase by 16c

dfprice.loc[dfprice['updated_at'].dt.tz_convert("Australia/Sydney").between(datetime(2026, 5, 1, tzinfo=timezone.utc), datetime(2026, 7, 2, tzinfo=timezone.utc)), 'price'] += 16

In [ ]:
dfprice.head()

In [ ]:
df91 = dfprice.loc[dfprice['fuel_type']=='U91',:]
df91mean = pd.DataFrame(df91.groupby('updated_at_dt')['price'].median())
df91mean.index = df91mean.index.astype('datetime64[ns]')
df91mean['price_change_pct'] = df91mean['price'].pct_change() * 100
df91mean['price_change'] = df91mean['price_change_pct'].diff()
df91mean.head()

In [ ]:
all_rows = select_supabase("market_data", columns="*", filters=None, batch_size=1000)
dfmarket = pd.DataFrame(all_rows)
dfmarket = dfmarket.pivot(index='date', columns='metric', values='value').reset_index()
dfmarket['date'] = pd.to_datetime(dfmarket['date'])
# fillna with the previous value
dfmarket.sort_values('date', inplace=True)
dfmarket.ffill(inplace=True)
dfmarket.tail()

In [ ]:
all_rows = select_supabase("servo_stations", columns="id, name, gcc_name21, sa4_name21", filters=None, batch_size=1000)
dfstations = pd.DataFrame(all_rows)
dfstations.head()

In [ ]:
plt.plot(dfmarket.date, dfmarket['mogas_95'], label='Mogas 95')
plt.plot(dfmarket.date, dfmarket['brent_crude'], label='brent_crude')
plt.plot(df91mean.index, df91mean['price'], label='U91 mean price')
plt.grid()
plt.legend()
plt.title('Mogas 95, Brent Crude, and U91 Mean Price Over Time')
plt.xlabel('Date')
plt.xticks(rotation=45)
plt.show()

In [ ]:
df91 = df91mean.merge(dfmarket, left_on='updated_at_dt', right_on='date', how='outer')


In [ ]:
# correlation between today's brent_crude and future price values
corr_rows = []

servo_rows = df91.loc[df91['price'].notna()].shape[0]

def correl_metric(df, metric, laglen):

    tmp = df[['date', 'price', metric]].sort_values('date').copy()
    for lag in range(0, laglen+1):
        shifted = tmp.copy()
        shifted[f'{metric}_lag'] = shifted[metric].shift(lag)

        corr = shifted[['price', f'{metric}_lag']].dropna().corr().iloc[0, 1]
        corr_rows.append({f'{metric}_lag': lag, 'correlation': corr})

    corr_df = pd.DataFrame(corr_rows)
    plt.figure(figsize=(8, 4))
    sns.lineplot(data=corr_df, x=f'{metric}_lag', y='correlation', marker='o')
    plt.axhline(0, color='black', linewidth=1)
    plt.title(f'Correlation of Price vs {metric} at Different Backward Offsets')
    plt.xlabel(f'{metric} Lag (days)')
    plt.ylabel('Correlation')
    plt.grid(True)
    plt.show()
    
correl_metric(df91, "brent_crude", 40)
correl_metric(df91, "mogas_95", 40)

In [ ]:
temp = df91.copy()#df91.loc[df91['price'].notna()].copy()
temp['brent_lag_30'] = temp['brent_crude'].shift(5)
temp.loc[temp['price'].notna(), ['price', 'brent_lag_30']].plot()
plt.grid()


In [ ]:
dfprice_geo = dfprice.merge(dfstations, left_on='station_id', right_on='id', how='left')


In [ ]:
temp = dfprice_geo.loc[dfprice_geo['fuel_type']=='U91',:].groupby(['gcc_name21','updated_at_dt'])['price'].median()

sns.lineplot(data=temp.reset_index(), x='updated_at_dt', y='price', hue='gcc_name21')
plt.title('Average U91 Price by GCCSA Over Time')
# Legend outside the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.show()

In [ ]:
temp = dfprice_geo.loc[dfprice_geo['fuel_type']=='U91',:].groupby(['sa4_name21','updated_at_dt'])['price'].median()

sns.lineplot(data=temp.reset_index(), x='updated_at_dt', y='price', hue='sa4_name21')
plt.title('Average U91 Price by SA4 Region Over Time')
# Legend outside the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.show()

In [ ]:
#df91['Weekday'] = df91.loc[df91['price'].notna(), 'date'].dt.weekday
df91['Weekday'] = df91['date'].dt.weekday
df91['WeekNumber'] = df91['date'].dt.isocalendar().week

In [ ]:
sns.lineplot(data=df91.loc[df91['price_change_pct'].notna()], x='Weekday', y='price_change_pct', hue='WeekNumber', palette='viridis')
plt.grid()

In [ ]:
df91.head()

In [ ]:
temp=df91.copy()

temp['brent_crude_lag22'] = temp['brent_crude'].shift(5)
temp['brent_change'] = temp['brent_crude'].diff()
sns.lineplot(temp.loc[temp['price'].notna()], x='date',y='brent_change')


In [ ]:
temp

In [ ]:
sns.lineplot(data = df91.loc[temp['price'].notna()], x='date',y='price_change', label='Price Change')
sns.lineplot(temp.loc[temp['price'].notna()], x='date',y='brent_change', label='Brent Change')
plt.grid()
plt.xticks(rotation=45)
plt.show()

In [ ]:
check = temp.iloc[-25:,:].copy()
check['Forcast'] = 0
check.iloc[0]['Forcast'] = check.iloc[0]['price']
check


In [ ]:
startprice = check.iloc[0]['price']
forecast = []
for i , row in check.iterrows():
    if len(forecast) == 0:
        forecast.append(startprice)
        continue
    forecast.append(forecast[-1] + row['brent_change'])
    
check['Forcast'] = forecast

sns.lineplot(data = check, x='date',y='price', label='Actual Price')
sns.lineplot(data = check, x='date',y='Forcast', label='Forecasted Price')
plt.grid()
plt.xticks(rotation=45)
plt.show()

In [ ]:
check.iloc[-2]['price']

In [ ]:
check.iloc[-22:]['brent_change']

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Use rows where target and predictors are available

df91['brent_crude_lag_22'] = df91['brent_crude'].shift(5)


model_df = df91[['price', 'brent_crude_lag_22', 'Weekday', 'WeekNumber']].dropna().copy()

X = model_df[['brent_crude_lag_22', 'Weekday', 'WeekNumber']]
y = model_df['price']

lr_model = LinearRegression()
lr_model.fit(X, y)

model_df['price_pred'] = lr_model.predict(X)

print(f"Rows used: {len(model_df)}")
print(f"Intercept: {lr_model.intercept_:.4f}")
print("Coefficients:")
print(pd.Series(lr_model.coef_, index=X.columns))

print(f"R²: {r2_score(y, model_df['price_pred']):.4f}")

model_df[['price', 'price_pred']].head()

In [ ]:

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Prepare time series: use df91 which has price and brent_crude aligned by date
arima_df = df91[['date', 'price', 'brent_crude']].dropna(subset=['date']).sort_values('date').copy()

# Create 21-day lagged brent_crude as exogenous variable
arima_df['brent_lag_21'] = arima_df['brent_crude'].shift(21)

# Drop rows missing either target or exogenous
arima_df = arima_df.dropna(subset=['price', 'brent_lag_21']).set_index('date')
arima_df.index.freq = 'D'

endog = arima_df['price']
exog  = arima_df[['brent_lag_21']]

print(f"Observations available: {len(arima_df)}")

# ARIMAX(1,1,1) — simpler order suited to limited data
model = SARIMAX(endog, exog=exog, order=(1, 1, 1), trend='c')
result = model.fit(disp=False)

print(result.summary())


In [ ]:

# In-sample fitted values and metrics — skip first point (ARIMA d=1 initialisation artifact)
fitted = result.fittedvalues.iloc[1:]
actual = arima_df['price'].iloc[1:]

mae  = mean_absolute_error(actual, fitted)
rmse = np.sqrt(mean_squared_error(actual, fitted))
ss_res = ((actual - fitted) ** 2).sum()
ss_tot = ((actual - actual.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot

print(f"Observations: {len(actual)}  (1 dropped for differencing initialisation)")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

plt.figure(figsize=(12, 4))
plt.plot(actual.index, actual, label='Actual', alpha=0.8)
plt.plot(fitted.index, fitted, label='ARIMAX(1,1,1) fitted', linestyle='--', alpha=0.8)
plt.title('ARIMAX(1,1,1) — Actual vs Fitted U91 Price (brent_crude lag 21d)')
plt.xlabel('Date')
plt.ylabel('Price (c/L)')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Residual plot
resid = result.resid.iloc[1:]
plt.figure(figsize=(12, 3))
plt.plot(resid.index, resid, marker='o', markersize=3)
plt.axhline(0, color='black', linewidth=1)
plt.title('ARIMAX(1,1,1) Residuals')
plt.xlabel('Date')
plt.ylabel('Residual')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
